In [ ]:

import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import shuffle



# Step 1: Load and preprocess the data

In [ ]:
data = pd.read_csv('data/final.csv')

In [ ]:
data.head()

In [ ]:
list(data['ProductName'].value_counts()[:10].keys())

In [ ]:
l = sorted(data['AreaName'].unique())
print(l)

In [ ]:
l = sorted(data['IndustryName'].unique())
print(l)

In [ ]:
from dnn1 import are_ind
list(are_ind('SOUTHLAND','Re-Work'))

In [ ]:
# Perform any necessary data cleaning and preprocessing here

# Step 2: Encode categorical variables
cat_vars = ['AreaName', 'IndustryName','ProductName']
label_encoders = {}
for var in cat_vars:
    label_encoders[var] = LabelEncoder()
    data[var] = label_encoders[var].fit_transform(data[var])


In [ ]:
data.head()

In [ ]:
# Step 3: Split the data into training and testing sets
X = data[['AreaName', 'IndustryName']].values
# X = data[cat_vars].values
y = data['ProductName'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
X

In [ ]:

# Step 4: Define the Softmax DNN model
num_features = 2
num_classes = len(np.unique(y))

model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(num_features,)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

In [ ]:
# Step 5: Model Training
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Shuffle the training data before each epoch
X_train, y_train = shuffle(X_train, y_train)

model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))


In [ ]:
# Step 6: Model Evaluation
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")


In [ ]:

# Step 7: Model Prediction
# Assuming you have new customer data in a variable called 'new_data'
new_data = np.array([['AUCKLAND SOUTH', 'STEEL FABRICATORS']])  # Replace with actual new data


In [ ]:
# Preprocess the new data
new_data_encoded = np.zeros((new_data.shape[0], num_features))
for i, var in enumerate(cat_vars[:2]):
    new_data_encoded[:, i] = label_encoders[var].transform(new_data[:, i])


In [ ]:

# Make predictions
predictions = model.predict(new_data_encoded)
recommended_product = np.argmax(predictions, axis=1)

# Decode the recommended product back to its original label
recommended_product_label = label_encoders['ProductName'].inverse_transform(recommended_product)
print(f"Recommended Product: {recommended_product_label}")


In [ ]:
predictions

In [ ]:
# Step 7: Model Prediction
# Assuming you have new customer data in a variable called 'new_data'
new_data = np.array([['AUCKLAND SOUTH', 'STEEL FABRICATORS']])  # Replace with actual new data

# Preprocess the new data
new_data_encoded = np.zeros((new_data.shape[0], num_features))
for i, var in enumerate(cat_vars[:2]):
    new_data_encoded[:, i] = label_encoders[var].transform(new_data[:, i])

# Make predictions
predictions = model.predict(new_data_encoded)

# Get the top 10 recommended products
top_10_indices = np.argsort(predictions, axis=1)[:, -10:]  # Indices of top 10 products for each customer
top_10_products = label_encoders['ProductName'].inverse_transform(top_10_indices.flatten())
top_10_products = top_10_products.reshape((-1, 10))

print(top_10_products[0])

print("Top 10 Recommended Products:")
for i, products in enumerate(top_10_products):
    print(f"Customer {i+1}: {products}")


In [ ]:
model.save('models/softmax.h5')

In [ ]:
from tensorflow import keras
m1 = keras.models.load_model('models/softmax.h5')

In [ ]:
m1